In [ ]:
import os
import tarfile
import pandas as pd
import numpy as np
from PIL import Image
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import torchvision.models as models
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==========================================
# 0. Reproducibility
# ==========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)


In [ ]:
set_seed(456)

In [ ]:
# ==========================================
# 1. Extract and Locate Dataset
# ==========================================
tar_file_path = '/content/drive/MyDrive/TDL/waterbirds_v1.0.tar.gz'
data_root_dir = '/content/waterbirds'

os.makedirs(data_root_dir, exist_ok=True)

print(f"Extracting {tar_file_path} to {data_root_dir}...")
with tarfile.open(tar_file_path, 'r:gz') as tar:
    tar.extractall(path=data_root_dir)

waterbirds_dir = None
for root, dirs, files in os.walk(data_root_dir):
    if 'metadata.csv' in files:
        waterbirds_dir = root
        break

if waterbirds_dir is None:
    raise FileNotFoundError("metadata.csv not found")

print(f"Dataset found at: {waterbirds_dir}")

Extracting /content/drive/MyDrive/TDL/waterbirds_v1.0.tar.gz to /content/waterbirds...


/tmp/ipykernel_7083/3672851327.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=data_root_dir)


Dataset found at: /content/waterbirds


In [ ]:
# ==========================================
# 2. Dataset Class
# ==========================================
class WaterbirdsRepoDataset(Dataset):
    def __init__(self, root_dir, split_name, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        metadata_df = pd.read_csv(os.path.join(root_dir, "metadata.csv"))

        split_map = {"train": 0, "val": 1, "test": 2}
        split_idx = split_map[split_name]

        self.metadata = metadata_df[metadata_df["split"] == split_idx].reset_index(drop=True)

        self.img_filenames = self.metadata["img_filename"].values
        self.targets = self.metadata["y"].values
        self.backgrounds = self.metadata["place"].values

        self.groups = []
        for t, p in zip(self.targets, self.backgrounds):
            self.groups.append(t * 2 + p)  # cleaner

    def __len__(self):
        return len(self.img_filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.img_filenames[idx])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, self.targets[idx], self.groups[idx], self.img_filenames[idx]

In [ ]:
# ==========================================
# 3. Transforms
# ==========================================
train_transform = transforms.Compose([
    transforms.RandomResizedCrop((224, 224), scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

In [ ]:
# ==========================================
# 4. DataLoaders
# ==========================================
train_data = WaterbirdsRepoDataset(waterbirds_dir, "train", transform=train_transform)
val_data   = WaterbirdsRepoDataset(waterbirds_dir, "val", transform=eval_transform)
test_data  = WaterbirdsRepoDataset(waterbirds_dir, "test", transform=eval_transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# ==========================================
# 4. Model Setup
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

model.fc = nn.Sequential(
    nn.Dropout(p=0.0),
    nn.Linear(model.fc.in_features, 2)
)

model = model.to(device)

epochs = 25
criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model.parameters(),
    lr=1e-3,
    momentum=0.9,
    weight_decay=1e-4
)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 204MB/s]


In [ ]:

# ==========================================
# 5. Training Loop
# ==========================================
print("Starting ERM Training...")

for epoch in range(epochs):
    model.train()

    train_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}]")

    for images, targets, groups, _ in loop:
        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

        loop.set_postfix({
            "loss": train_loss / (loop.n + 1),
            "acc": correct / total
        })

    # ================= VALIDATION =================
    model.eval()

    group_correct = {0: 0, 1: 0, 2: 0, 3: 0}
    group_total   = {0: 0, 1: 0, 2: 0, 3: 0}

    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, targets, groups, _ in val_loader:
            images, targets = images.to(device), targets.to(device)
            groups = groups.to(device)

            logits = model(images)
            preds = torch.argmax(logits, dim=1)

            val_correct += (preds == targets).sum().item()
            val_total += targets.size(0)

            for i in range(len(targets)):
                g = groups[i].item()
                group_total[g] += 1
                if preds[i] == targets[i]:
                    group_correct[g] += 1

    group_accs = {
        g: group_correct[g] / (group_total[g] + 1e-8)
        for g in range(4)
    }

    wga = min(group_accs.values())
    avg_acc = val_correct / val_total

    print(f"Epoch {epoch+1:02d}/{epochs} | Train Acc: {correct/total:.4f} | Val Acc: {avg_acc:.4f} | Val WGA: {wga:.4f}")


Starting ERM Training...


Epoch [1/25]: 100%|██████████| 150/150 [00:54<00:00,  2.76it/s, loss=0.181, acc=0.932]


Epoch 01/25 | Train Acc: 0.9316 | Val Acc: 0.8307 | Val WGA: 0.4887


Epoch [2/25]: 100%|██████████| 150/150 [00:53<00:00,  2.81it/s, loss=0.0736, acc=0.976]


Epoch 02/25 | Train Acc: 0.9764 | Val Acc: 0.8582 | Val WGA: 0.5414


...

Epoch 23/25 | Train Acc: 1.0000 | Val Acc: 0.8874 | Val WGA: 0.7293


Epoch [24/25]: 100%|██████████| 150/150 [00:52<00:00,  2.83it/s, loss=0.00131, acc=1]


Epoch 24/25 | Train Acc: 0.9998 | Val Acc: 0.8891 | Val WGA: 0.6842


Epoch [25/25]: 100%|██████████| 150/150 [00:52<00:00,  2.84it/s, loss=0.000991, acc=1]


Epoch 25/25 | Train Acc: 1.0000 | Val Acc: 0.8907 | Val WGA: 0.6767


In [ ]:
# ==========================================
# 6. FINAL TEST EVALUATION
# ==========================================
print("\nFinal TEST Evaluation...")

model.eval()

group_correct = {0: 0, 1: 0, 2: 0, 3: 0}
group_total   = {0: 0, 1: 0, 2: 0, 3: 0}

test_correct = 0
test_total = 0

with torch.no_grad():
    for images, targets, groups, _ in test_loader:
        images, targets = images.to(device), targets.to(device)
        groups = groups.to(device)

        logits = model(images)
        preds = torch.argmax(logits, dim=1)

        test_correct += (preds == targets).sum().item()
        test_total += targets.size(0)

        for i in range(len(targets)):
            g = groups[i].item()
            group_total[g] += 1
            if preds[i] == targets[i]:
                group_correct[g] += 1

group_accs = {
    g: group_correct[g] / (group_total[g] + 1e-8)
    for g in range(4)
}

test_wga = min(group_accs.values())
test_avg_acc = test_correct / test_total

print("\nTEST RESULTS")
print(f"Avg Accuracy: {test_avg_acc:.4f}")
print(f"Group Accuracies: {group_accs}")
print(f"WGA: {test_wga:.4f}")

# ==========================================
# 7. Per-Sample Logging (IMPORTANT)
# ==========================================
print("\nExtracting per-sample predictions from Test Set...")

test_predictions_log = []

with torch.no_grad():
    for images, targets, groups, img_paths in test_loader:
        images, targets = images.to(device), targets.to(device)
        groups = groups.to(device)

        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)

        for i in range(len(targets)):
            test_predictions_log.append({
                "image_id": img_paths[i],
                "label": targets[i].item(),
                "group": groups[i].item(),
                "prediction": preds[i].item(),
                "confidence": probs[i][preds[i]].item(),
                "logit_class_0": logits[i][0].item(),
                "logit_class_1": logits[i][1].item()
            })

# Save outputs
torch.save(model.state_dict(), "erm_final_model.pth")
pd.DataFrame(test_predictions_log).to_csv("erm_test_predictions.csv", index=False)

print("Saved final ERM model and predictions.")


Final TEST Evaluation...

TEST RESULTS
Avg Accuracy: 0.8932
Group Accuracies: {0: 0.9964523281552264, 1: 0.8292682926792494, 2: 0.6838006230423084, 3: 0.9641744548136422}
WGA: 0.6838

Extracting per-sample predictions from Test Set...
Saved final ERM model and predictions.


In [ ]:
# ==========================================
# FREE LUNCH (95/5 SPLIT)
# ==========================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
import torchvision.models as models
import numpy as np
import pandas as pd
from tqdm import tqdm

print("\n--- Free Lunch: 95/5 Split ---")

# ==========================================
# 1. 95/5 SPLIT
# ==========================================
total_train = len(train_data)
split_95_len = int(0.95 * total_train)
split_5_len = total_train - split_95_len

train_95_ds, train_5_ds = random_split(
    train_data,
    [split_95_len, split_5_len],
    generator=torch.Generator().manual_seed(42)
)

train_95_loader = DataLoader(train_95_ds, batch_size=32, shuffle=True, num_workers=2)

print(f"95% split: {split_95_len}, 5% split: {split_5_len}")

# ==========================================
# 2. TRAIN BACKBONE (95%)
# ==========================================
print("\n[Phase 1] Training backbone on 95%")

model_95 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
in_features = model_95.fc.in_features

model_95.fc = nn.Sequential(
    nn.Dropout(0.0),
    nn.Linear(in_features, 2)
)

model_95 = model_95.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_95.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)

epochs_95 = 25

for epoch in range(epochs_95):
    model_95.train()
    total_loss = 0
    correct = 0
    total = 0

    loop = tqdm(train_95_loader, desc=f"Backbone Epoch [{epoch+1}/{epochs_95}]")

    for images, targets, _, _ in loop:
        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()
        logits = model_95(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

        loop.set_postfix({
            "loss": total_loss / (loop.n + 1),
            "acc": correct / total
        })

    print(f"Epoch {epoch+1}/{epochs_95} | Loss: {total_loss/len(train_95_loader):.4f} | Acc: {correct/total:.4f}")

# ==========================================
# 3. CLASS-BALANCED SAMPLER (5%)
# ==========================================
print("\n[Phase 2] Class-balanced retraining")

targets_5 = [train_data[i][1] for i in train_5_ds.indices]
class_counts = np.bincount(targets_5)

class_weights = 1.0 / class_counts
sample_weights = np.array([class_weights[y] for y in targets_5])

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_5_loader = DataLoader(train_5_ds, batch_size=32, sampler=sampler, num_workers=2)

# ==========================================
# 4. FREEZE + RETRAIN LAST LAYER
# ==========================================
for param in model_95.parameters():
    param.requires_grad = False

model_95.fc = nn.Sequential(
    nn.Dropout(0.0),
    nn.Linear(in_features, 2)
)

model_95 = model_95.to(device)

optimizer_fc = optim.SGD(model_95.fc.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)

finetune_steps = 250
model_95.train()

step = 0
train_iter = iter(train_5_loader)

print("\nRetraining last layer...")

pbar = tqdm(total=finetune_steps, desc="Last Layer Finetuning")

while step < finetune_steps:
    try:
        images, targets, _, _ = next(train_iter)
    except StopIteration:
        train_iter = iter(train_5_loader)
        images, targets, _, _ = next(train_iter)

    images, targets = images.to(device), targets.to(device)

    optimizer_fc.zero_grad()
    logits = model_95(images)
    loss = criterion(logits, targets)
    loss.backward()
    optimizer_fc.step()

    step += 1

    pbar.update(1)
    pbar.set_postfix({"loss": loss.item()})

pbar.close()

# ==========================================
# 5. EVALUATION (TEST)
# ==========================================
print("\n[Phase 3] Evaluating Free Lunch model")

model_95.eval()

group_correct = {0: 0, 1: 0, 2: 0, 3: 0}
group_total = {0: 0, 1: 0, 2: 0, 3: 0}

test_correct = 0
test_total = 0

logs = []

with torch.no_grad():
    for images, targets, groups, paths in test_loader:
        images, targets = images.to(device), targets.to(device)
        groups = groups.to(device)

        logits = model_95(images)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)

        test_correct += (preds == targets).sum().item()
        test_total += targets.size(0)

        for i in range(len(targets)):
            g = groups[i].item()
            group_total[g] += 1
            if preds[i] == targets[i]:
                group_correct[g] += 1

            logs.append({
                "image_id": paths[i],
                "label": targets[i].item(),
                "group": g,
                "fl_prediction": preds[i].item(),
                "fl_confidence": probs[i][preds[i]].item()
            })

group_accs = {
    g: group_correct[g] / (group_total[g] + 1e-8)
    for g in range(4)
}

wga = min(group_accs.values())
avg_acc = test_correct / test_total

print("\nFREE LUNCH RESULTS")
print(f"Avg Accuracy: {avg_acc:.4f}")
print(f"WGA: {wga:.4f}")
print(f"Group Accuracies: {group_accs}")

# ==========================================
# SAVE
# ==========================================
torch.save(model_95.state_dict(), "free_lunch_model.pth")
pd.DataFrame(logs).to_csv("free_lunch_test_predictions.csv", index=False)

print("Saved Free Lunch model + predictions")


--- Free Lunch: 95/5 Split ---
95% split: 4555, 5% split: 240

[Phase 1] Training backbone on 95%


Backbone Epoch [1/25]: 100%|██████████| 143/143 [00:46<00:00,  3.09it/s, loss=0.184, acc=0.925]


Epoch 1/25 | Loss: 0.1839 | Acc: 0.9247


Backbone Epoch [2/25]: 100%|██████████| 143/143 [00:47<00:00,  3.04it/s, loss=0.081, acc=0.971]


Epoch 2/25 | Loss: 0.0810 | Acc: 0.9706


...

Backbone Epoch [25/25]: 100%|██████████| 143/143 [00:48<00:00,  2.97it/s, loss=0.00132, acc=1]


Epoch 25/25 | Loss: 0.0013 | Acc: 1.0000

[Phase 2] Class-balanced retraining

Retraining last layer...


Last Layer Finetuning: 100%|██████████| 250/250 [00:46<00:00,  5.38it/s, loss=0.0179]


[Phase 3] Evaluating Free Lunch model



FREE LUNCH RESULTS
Avg Accuracy: 0.8550
WGA: 0.7144
Group Accuracies: {0: 0.9937915742749721, 1: 0.7144124168482732, 2: 0.7461059189914937, 3: 0.9704049844085607}
Saved Free Lunch model + predictions


Free Lunch
- Use 95% to train model
- 5% random is used to retrain classifier
- Classifier training samples are class balanced - equal number of landbirds and waterbirds - ensures representation without needing group labels

DFR
- classifier retraining happenes with group balancing
- need to know groups (landbird-land, landbird-water) which is not usually available in real deployment scenarios
- uses a 5 or 10% group balanced held-out split(train+val split, val not used to train) or subset(of training data) for retraining of classifier
- each grp has equal number of samples in the classifier retraining data

FL WGA > ERM WGA

Seed 123 ERM

In [ ]:

# ==========================================
# 5. Training Loop
# ==========================================
print("Starting ERM Training...")

for epoch in range(epochs):
    model.train()

    train_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}]")

    for images, targets, groups, _ in loop:
        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

        loop.set_postfix({
            "loss": train_loss / (loop.n + 1),
            "acc": correct / total
        })

    # ================= VALIDATION =================
    model.eval()

    group_correct = {0: 0, 1: 0, 2: 0, 3: 0}
    group_total   = {0: 0, 1: 0, 2: 0, 3: 0}

    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, targets, groups, _ in val_loader:
            images, targets = images.to(device), targets.to(device)
            groups = groups.to(device)

            logits = model(images)
            preds = torch.argmax(logits, dim=1)

            val_correct += (preds == targets).sum().item()
            val_total += targets.size(0)

            for i in range(len(targets)):
                g = groups[i].item()
                group_total[g] += 1
                if preds[i] == targets[i]:
                    group_correct[g] += 1

    group_accs = {
        g: group_correct[g] / (group_total[g] + 1e-8)
        for g in range(4)
    }

    wga = min(group_accs.values())
    avg_acc = val_correct / val_total

    print(f"Epoch {epoch+1:02d}/{epochs} | Train Acc: {correct/total:.4f} | Val Acc: {avg_acc:.4f} | Val WGA: {wga:.4f}")


Starting ERM Training...


Epoch [1/25]: 100%|██████████| 150/150 [00:48<00:00,  3.10it/s, loss=0.185, acc=0.927]


Epoch 01/25 | Train Acc: 0.9272 | Val Acc: 0.8165 | Val WGA: 0.5489


Epoch [2/25]: 100%|██████████| 150/150 [00:51<00:00,  2.92it/s, loss=0.0727, acc=0.975]


Epoch 02/25 | Train Acc: 0.9754 | Val Acc: 0.8849 | Val WGA: 0.4586


...

Epoch 23/25 | Train Acc: 0.9990 | Val Acc: 0.8791 | Val WGA: 0.6241


Epoch [24/25]: 100%|██████████| 150/150 [00:54<00:00,  2.76it/s, loss=0.00252, acc=0.999]


Epoch 24/25 | Train Acc: 0.9992 | Val Acc: 0.8557 | Val WGA: 0.7210


Epoch [25/25]: 100%|██████████| 150/150 [00:54<00:00,  2.76it/s, loss=0.0017, acc=1]


Epoch 25/25 | Train Acc: 0.9998 | Val Acc: 0.8924 | Val WGA: 0.6165


In [ ]:
# ==========================================
# 6. FINAL TEST EVALUATION
# ==========================================
print("\nFinal TEST Evaluation...")

model.eval()

group_correct = {0: 0, 1: 0, 2: 0, 3: 0}
group_total   = {0: 0, 1: 0, 2: 0, 3: 0}

test_correct = 0
test_total = 0

with torch.no_grad():
    for images, targets, groups, _ in test_loader:
        images, targets = images.to(device), targets.to(device)
        groups = groups.to(device)

        logits = model(images)
        preds = torch.argmax(logits, dim=1)

        test_correct += (preds == targets).sum().item()
        test_total += targets.size(0)

        for i in range(len(targets)):
            g = groups[i].item()
            group_total[g] += 1
            if preds[i] == targets[i]:
                group_correct[g] += 1

group_accs = {
    g: group_correct[g] / (group_total[g] + 1e-8)
    for g in range(4)
}

test_wga = min(group_accs.values())
test_avg_acc = test_correct / test_total

print("\nTEST RESULTS")
print(f"Avg Accuracy: {test_avg_acc:.4f}")
print(f"Group Accuracies: {group_accs}")
print(f"WGA: {test_wga:.4f}")

# ==========================================
# 7. Per-Sample Logging (IMPORTANT)
# ==========================================
print("\nExtracting per-sample predictions from Test Set...")

test_predictions_log = []

with torch.no_grad():
    for images, targets, groups, img_paths in test_loader:
        images, targets = images.to(device), targets.to(device)
        groups = groups.to(device)

        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)

        for i in range(len(targets)):
            test_predictions_log.append({
                "image_id": img_paths[i],
                "label": targets[i].item(),
                "group": groups[i].item(),
                "prediction": preds[i].item(),
                "confidence": probs[i][preds[i]].item(),
                "logit_class_0": logits[i][0].item(),
                "logit_class_1": logits[i][1].item()
            })

# Save outputs
torch.save(model.state_dict(), "erm_final_model.pth")
pd.DataFrame(test_predictions_log).to_csv("erm_test_predictions.csv", index=False)

print("Saved final ERM model and predictions.")


Final TEST Evaluation...

TEST RESULTS
Avg Accuracy: 0.9014
Group Accuracies: {0: 0.998226164075396, 1: 0.8603104212822159, 2: 0.6510903426689861, 3: 0.956386292819994}
WGA: 0.6511

Extracting per-sample predictions from Test Set...
Saved final ERM model and predictions.


Seed 456 ERM

In [ ]:

# ==========================================
# 5. Training Loop
# ==========================================
print("Starting ERM Training...")

for epoch in range(epochs):
    model.train()

    train_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}]")

    for images, targets, groups, _ in loop:
        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

        loop.set_postfix({
            "loss": train_loss / (loop.n + 1),
            "acc": correct / total
        })

    # ================= VALIDATION =================
    model.eval()

    group_correct = {0: 0, 1: 0, 2: 0, 3: 0}
    group_total   = {0: 0, 1: 0, 2: 0, 3: 0}

    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, targets, groups, _ in val_loader:
            images, targets = images.to(device), targets.to(device)
            groups = groups.to(device)

            logits = model(images)
            preds = torch.argmax(logits, dim=1)

            val_correct += (preds == targets).sum().item()
            val_total += targets.size(0)

            for i in range(len(targets)):
                g = groups[i].item()
                group_total[g] += 1
                if preds[i] == targets[i]:
                    group_correct[g] += 1

    group_accs = {
        g: group_correct[g] / (group_total[g] + 1e-8)
        for g in range(4)
    }

    wga = min(group_accs.values())
    avg_acc = val_correct / val_total

    print(f"Epoch {epoch+1:02d}/{epochs} | Train Acc: {correct/total:.4f} | Val Acc: {avg_acc:.4f} | Val WGA: {wga:.4f}")


In [ ]:
# ==========================================
# 6. FINAL TEST EVALUATION
# ==========================================
print("\nFinal TEST Evaluation...")

model.eval()

group_correct = {0: 0, 1: 0, 2: 0, 3: 0}
group_total   = {0: 0, 1: 0, 2: 0, 3: 0}

test_correct = 0
test_total = 0

with torch.no_grad():
    for images, targets, groups, _ in test_loader:
        images, targets = images.to(device), targets.to(device)
        groups = groups.to(device)

        logits = model(images)
        preds = torch.argmax(logits, dim=1)

        test_correct += (preds == targets).sum().item()
        test_total += targets.size(0)

        for i in range(len(targets)):
            g = groups[i].item()
            group_total[g] += 1
            if preds[i] == targets[i]:
                group_correct[g] += 1

group_accs = {
    g: group_correct[g] / (group_total[g] + 1e-8)
    for g in range(4)
}

test_wga = min(group_accs.values())
test_avg_acc = test_correct / test_total

print("\nTEST RESULTS")
print(f"Avg Accuracy: {test_avg_acc:.4f}")
print(f"Group Accuracies: {group_accs}")
print(f"WGA: {test_wga:.4f}")

# ==========================================
# 7. Per-Sample Logging (IMPORTANT)
# ==========================================
print("\nExtracting per-sample predictions from Test Set...")

test_predictions_log = []

with torch.no_grad():
    for images, targets, groups, img_paths in test_loader:
        images, targets = images.to(device), targets.to(device)
        groups = groups.to(device)

        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)

        for i in range(len(targets)):
            test_predictions_log.append({
                "image_id": img_paths[i],
                "label": targets[i].item(),
                "group": groups[i].item(),
                "prediction": preds[i].item(),
                "confidence": probs[i][preds[i]].item(),
                "logit_class_0": logits[i][0].item(),
                "logit_class_1": logits[i][1].item()
            })

# Save outputs
torch.save(model.state_dict(), "erm_final_model_seed456.pth")
pd.DataFrame(test_predictions_log).to_csv("erm_test_predictions_seed456.csv", index=False)

print("Saved final ERM model and predictions.")

Seed 456 ERM

In [ ]:

# ==========================================
# 5. Training Loop
# ==========================================
print("Starting ERM Training...")

for epoch in range(epochs):
    model.train()

    train_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}]")

    for images, targets, groups, _ in loop:
        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

        loop.set_postfix({
            "loss": train_loss / (loop.n + 1),
            "acc": correct / total
        })

    # ================= VALIDATION =================
    model.eval()

    group_correct = {0: 0, 1: 0, 2: 0, 3: 0}
    group_total   = {0: 0, 1: 0, 2: 0, 3: 0}

    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, targets, groups, _ in val_loader:
            images, targets = images.to(device), targets.to(device)
            groups = groups.to(device)

            logits = model(images)
            preds = torch.argmax(logits, dim=1)

            val_correct += (preds == targets).sum().item()
            val_total += targets.size(0)

            for i in range(len(targets)):
                g = groups[i].item()
                group_total[g] += 1
                if preds[i] == targets[i]:
                    group_correct[g] += 1

    group_accs = {
        g: group_correct[g] / (group_total[g] + 1e-8)
        for g in range(4)
    }

    wga = min(group_accs.values())
    avg_acc = val_correct / val_total

    print(f"Epoch {epoch+1:02d}/{epochs} | Train Acc: {correct/total:.4f} | Val Acc: {avg_acc:.4f} | Val WGA: {wga:.4f}")


Starting ERM Training...


Epoch [1/25]: 100%|██████████| 150/150 [00:50<00:00,  2.98it/s, loss=0.184, acc=0.929]


Epoch 01/25 | Train Acc: 0.9289 | Val Acc: 0.7873 | Val WGA: 0.5263


Epoch [2/25]: 100%|██████████| 150/150 [00:47<00:00,  3.17it/s, loss=0.0765, acc=0.971]


Epoch 02/25 | Train Acc: 0.9712 | Val Acc: 0.8557 | Val WGA: 0.6316


...

Epoch 23/25 | Train Acc: 0.9994 | Val Acc: 0.8866 | Val WGA: 0.7744


Epoch [24/25]: 100%|██████████| 150/150 [00:49<00:00,  3.02it/s, loss=0.00147, acc=1]


Epoch 24/25 | Train Acc: 1.0000 | Val Acc: 0.8882 | Val WGA: 0.6992


Epoch [25/25]: 100%|██████████| 150/150 [00:49<00:00,  3.01it/s, loss=0.00115, acc=1]


Epoch 25/25 | Train Acc: 0.9998 | Val Acc: 0.8941 | Val WGA: 0.7444


In [ ]:
# ==========================================
# 6. FINAL TEST EVALUATION
# ==========================================
print("\nFinal TEST Evaluation...")

model.eval()

group_correct = {0: 0, 1: 0, 2: 0, 3: 0}
group_total   = {0: 0, 1: 0, 2: 0, 3: 0}

test_correct = 0
test_total = 0

with torch.no_grad():
    for images, targets, groups, _ in test_loader:
        images, targets = images.to(device), targets.to(device)
        groups = groups.to(device)

        logits = model(images)
        preds = torch.argmax(logits, dim=1)

        test_correct += (preds == targets).sum().item()
        test_total += targets.size(0)

        for i in range(len(targets)):
            g = groups[i].item()
            group_total[g] += 1
            if preds[i] == targets[i]:
                group_correct[g] += 1

group_accs = {
    g: group_correct[g] / (group_total[g] + 1e-8)
    for g in range(4)
}

test_wga = min(group_accs.values())
test_avg_acc = test_correct / test_total

print("\nTEST RESULTS")
print(f"Avg Accuracy: {test_avg_acc:.4f}")
print(f"Group Accuracies: {group_accs}")
print(f"WGA: {test_wga:.4f}")

# ==========================================
# 7. Per-Sample Logging (IMPORTANT)
# ==========================================
print("\nExtracting per-sample predictions from Test Set...")

test_predictions_log = []

with torch.no_grad():
    for images, targets, groups, img_paths in test_loader:
        images, targets = images.to(device), targets.to(device)
        groups = groups.to(device)

        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)

        for i in range(len(targets)):
            test_predictions_log.append({
                "image_id": img_paths[i],
                "label": targets[i].item(),
                "group": groups[i].item(),
                "prediction": preds[i].item(),
                "confidence": probs[i][preds[i]].item(),
                "logit_class_0": logits[i][0].item(),
                "logit_class_1": logits[i][1].item()
            })

# Save outputs
torch.save(model.state_dict(), "erm_final_model_seed456.pth")
pd.DataFrame(test_predictions_log).to_csv("erm_test_predictions_seed456.csv", index=False)

print("Saved final ERM model and predictions.")


Final TEST Evaluation...

TEST RESULTS
Avg Accuracy: 0.8885
Group Accuracies: {0: 0.996008869175184, 1: 0.8039911308168338, 2: 0.7305295950041973, 3: 0.9657320872123718}
WGA: 0.7305

Extracting per-sample predictions from Test Set...
Saved final ERM model and predictions.
